In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Generate flow that highly depends on yesterday's flow (rho=0.8)
import scipy.signal

np.random.seed(42)
T = 100
rainfall = np.random.uniform(0, 20, T)
runoff_coeff = 0.5

# Prepare today's input (rainfall effect + random noise).
noise = np.random.normal(0, 1, T)
exog = runoff_coeff * rainfall + noise
exog[0] = 0  # Anchor first day so we match the base flow cleanly

# lfilter pushes the recursion to C: y[t] = 0.8*y[t-1] + exog[t]
# The 'a' array defines the left side of the equation: y[t] - 0.8*y[t-1] = exog[t]
flow_centered = scipy.signal.lfilter(b=[1.0], a=[1.0, -0.8], x=exog)
flow_true = flow_centered + 10  # Shift back up to resting baseline (10)

with pm.Model() as model_static:

    b_rain = pm.HalfNormal("b_rain", sigma=2)
    intercept = pm.Normal("intercept", mu=10, sigma=5)
    sigma = pm.HalfNormal("sigma", sigma=5)
    mu = pm.Deterministic("mu", intercept + b_rain * rainfall)
    y = pm.Normal("y", mu=mu, sigma=sigma, observed=flow_true)
    trace_static = pm.sample(1000, tune=1000, cores=1, random_seed=42)

    prior_checks = pm.sample_prior_predictive(draws=1000, random_seed=42)

    trace_static = pm.sample(draws=2000, tune=1000, chains=2, random_seed=42, progressbar=False)

static_preds = trace_static.posterior["mu"].mean(dim=["chain", "draw"])
residuals = flow_true - static_preds

plt.acorr(residuals, maxlags=20)
plt.title("M2 Static Model Residuals: MASSIVE AUTOCORRELATION")
plt.show()